# Project Aftershock — Phase 2: Data Understanding

Regional Seismic Risk Triage. This notebook explores the raw USGS earthquake
catalog pull before any cleaning happens (that's Phase 3, in `src/pipeline.py`).

Uses only `requests`, `json`, `csv`, `pathlib` and native loops — no pandas/numpy.

In [1]:
import json
import sys
from pathlib import Path

sys.path.insert(0, "..")
from src.pipeline import (
    fetch_usgs_catalog,
    walk_record,
    _running_min_max_mean,
    null_rate_report,
    type_breakdown,
    QUALITY_FLAG_FIELDS,
)

## 1. Fetch the live catalog

One `requests.get()` call, wrapped in try/except. Falls back to a local cache only if the live call fails (e.g. in a network-restricted sandbox).

In [2]:
raw_features = fetch_usgs_catalog(cache_path="../data/raw/usgs_sample_cache.json")
print(f"Fetched {len(raw_features)} raw features.")

Live USGS request failed (403 Client Error: Forbidden for url: https://earthquake.usgs.gov/fdsnws/event/1/query?format=geojson&starttime=2026-08-15&endtime=2026-08-29&minmagnitude=2.5).
Falling back to cached response at ../data/raw/usgs_sample_cache.json.
Fetched 44 raw features.


## 2. Structural audit

Recursively walk one record and print the type of every leaf value alongside its path — this is where `properties` vs `geometry` nesting becomes clear.

In [3]:
print(json.dumps(raw_features[0], indent=2))

{
  "type": "Feature",
  "id": "ci41538696",
  "properties": {
    "mag": 0.24,
    "place": "12 km WNW of Anza, CA",
    "time": 1787998233170,
    "felt": null,
    "cdi": null,
    "mmi": null,
    "alert": null,
    "tsunami": 0,
    "sig": 2,
    "net": "ci",
    "nst": 25,
    "dmin": 0.03358,
    "rms": 0.11,
    "gap": 43,
    "magType": "ml",
    "type": "earthquake"
  },
  "geometry": {
    "type": "Point",
    "coordinates": [
      -116.79833333333,
      33.576,
      7.67
    ]
  }
}


In [4]:
walk_record(raw_features[0])

root.type: str = 'Feature'
root.id: str = 'ci41538696'
root.properties.mag: float = 0.24
root.properties.place: str = '12 km WNW of Anza, CA'
root.properties.time: int = 1787998233170
root.properties.felt: NoneType = None
root.properties.cdi: NoneType = None
root.properties.mmi: NoneType = None
root.properties.alert: NoneType = None
root.properties.tsunami: int = 0
root.properties.sig: int = 2
root.properties.net: str = 'ci'
root.properties.nst: int = 25
root.properties.dmin: float = 0.03358
root.properties.rms: float = 0.11
root.properties.gap: int = 43
root.properties.magType: str = 'ml'
root.properties.type: str = 'earthquake'
root.geometry.type: str = 'Point'
root.geometry.coordinates[0]: float = -116.79833333333


## 3. Native-loop EDA

Min/max/mean for `mag` and `depth_km`, computed with single-pass running accumulators — no `min()`/`max()`/`sum()` builtins. Records where the field is `None` are skipped before they reach the accumulator.

In [5]:
mags = [f["properties"].get("mag") for f in raw_features]
depths = [f["geometry"]["coordinates"][2] if f.get("geometry") else None for f in raw_features]

mag_min, mag_max, mag_mean = _running_min_max_mean(mags)
depth_min, depth_max, depth_mean = _running_min_max_mean(depths)

print(f"mag:      min={mag_min}, max={mag_max}, mean={mag_mean:.3f}")
print(f"depth_km: min={depth_min}, max={depth_max}, mean={depth_mean:.3f}")

mag:      min=0.24, max=5.0, mean=1.976
depth_km: min=-0.001, max=602.27, mean=31.835


## 4. Quality verification

Null-rate for the fields the brief flags as genuinely sparse (`felt`, `cdi`, `mmi`, `alert`, `nst`, `dmin`, `gap`), plus the `type` breakdown — this is where explosion/quarry-blast contamination becomes visible.

In [6]:
for field, pct in null_rate_report(raw_features, QUALITY_FLAG_FIELDS).items():
    print(f"{field}: {pct:.1f}% null")

felt: 95.5% null
cdi: 95.5% null
mmi: 95.5% null
alert: 88.6% null
nst: 0.0% null
dmin: 4.5% null
gap: 0.0% null


In [7]:
for t, count in type_breakdown(raw_features).items():
    print(f"{t}: {count}")

earthquake: 42
quarry blast: 1
explosion: 1


## 5. Takeaways feeding into Phase 3

- `felt`/`cdi`/`mmi`/`alert` are null on the large majority of records (mostly small, automatically-processed events) — confirms the brief's messiness warning.
- `type` is not always `"earthquake"` — `quarry blast` and `explosion` records are mixed in and must be filtered out in Phase 3.
- `place` will need defensive splitting on `" of "` since not every string contains that pattern (offshore events).
- `depth_km` can be negative (above-reference-point quirk) — not something to "fix", just something Phase 3's depth-bucketing needs to handle gracefully.

All of this is handled in `src/pipeline.py`, which is fully runnable standalone: `python src/pipeline.py`.